# Pixels to Predictions: CS-GY 6953 (Deep Learning) Kaggle Competition
## MODEL 8 (Ensemble model)
### Author: Mariia Onokhina

---
### Experiment Results
- Model: 

---
**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

## Installation of Libraries 

Before installing any libraries, I create a new Conda environment and add it to Jupyter Notebook to ensure I start from a clean slate and that the code is reproducible.

**Run the following code in a terminal if you'd like to start fresh with a new environment:**
```bash
conda create -n pixels-to-predictions python=3.10
conda activate pixels-to-predictions
conda install -c conda-forge notebook
conda install -c conda-forge ipykernel
python -m ipykernel install --user --name pixels-to-predictions --display-name "Pixels-to-predictions DL"
```

IMPORTANT: Manually change the Kernel in Jupyter Notebook in VS Code or Jupyter Lab to "Pixels-to-predictions DL".

In [1]:
# Uncomment this cell to install the necessary Python packages.
import sys
print("Python:", sys.executable)
!{sys.executable} -m pip install -q transformers==4.57.6 peft==0.18.1 kaggle matplotlib scikit-learn pandas numpy ipywidgets jupyterlab_widgets bitsandbytes accelerate datasets pillow --quiet

Python: /home/devvingduo/miniforge3/envs/pixels-to-predictions/bin/python


---
## Imports & Configuration

In [2]:
# Imports & Configuration
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import ast
from transformers import AutoProcessor, AutoModelForVision2Seq

# For LoRa fine-tuning
from peft import LoraConfig, get_peft_model, PeftModel
from tqdm.auto import tqdm   # For progress bar
from itertools import combinations
import gc
from collections import Counter

import pickle
from PIL import Image

In [3]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [4]:
# This code makes sure it uses only 1 GPU of my choice
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [5]:
# Paths 
DATA_DIR = Path("../data")
IMAGE_ROOT = Path("../data/images")

# Model
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# Basic Settings
IMG_SIZE = 512

# This run will be entirely with mean scoring
SCORE_MODE = "mean"

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 3090


---
### Load and Preprocess Data

Download data from https://www.kaggle.com/competitions/pixels-to-predictions/data via Kaggle CLI.

For this, you need a Legacy API key which you can create here: https://www.kaggle.com/settings.

When you create a new key, it will download a ```kaggle.json```. 

In your terminal, run:
```bash
mkdir -p ~/.kaggle
mv <your_downloads_folder> ~/.kaggle/kaggle.json
chmod 600 ~/.kaggle/kaggle.json
```

Verify that it worked by running: 
```bash
ls -la ~/.kaggle
```

I have to add it manually because I'm working via SSH into my Linux server machine.

In [7]:
# Uncomment this cell to download the data in a .zip file
#!kaggle competitions download -c pixels-to-predictions

In [8]:
# Uncomment this cell to unzip the data into "data" folder
#!unzip pixels-to-predictions.zip -d data

In [9]:
# Load CSVs
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

Inspect the data. 

In [10]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3109 entries, 0 to 3108
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           3109 non-null   object
 1   image_path   3109 non-null   object
 2   question     3109 non-null   object
 3   choices      3109 non-null   object
 4   num_choices  3109 non-null   int64 
 5   answer       3109 non-null   int64 
 6   hint         2385 non-null   object
 7   lecture      2669 non-null   object
 8   solution     2580 non-null   object
 9   task         3109 non-null   object
 10  grade        3109 non-null   object
 11  subject      3109 non-null   object
 12  topic        3109 non-null   object
 13  category     3109 non-null   object
 14  skill        3109 non-null   object
dtypes: int64(2), object(13)
memory usage: 364.5+ KB


In [11]:
# Function to parse the choices column using ast module (Abstract Syntax Tree)
def parse_choices(x):
    # If x is already a list, return it
    if isinstance(x, list):
        return x
    # If x is a string, parse it using ast.literal_eval
    return ast.literal_eval(x)

The ```choices``` column is a JSON string, so we parse it using the function above.

In [12]:
for df in [train_df, val_df, test_df]:
    df["choices_list"] = df["choices"].apply(parse_choices)

In [34]:
def load_image(path):
    image = Image.open(path).convert("RGB")
    image = image.resize((512, 512)) 
    return image

---
### Prompt

Keep `build_prompt_model2_imagecareful` which is a variation of Model 2's prompt. 

We want to see whether including lecture field in the prompt improves the score. It's very long so we truncate it.

In [14]:
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.lower() in ["nan", "none", "null"]:
        return ""
    return x

In [15]:
def truncate_words(x, max_words=180):
    x = clean_text(x)
    if not x:
        return ""
    words = x.split()
    return " ".join(words[:max_words])

In [16]:
PROMPT_STYLE = "lecture_180"


def build_prompt(row):
    choices = row["choices_list"]

    choice_lines = "\n".join(
        [f"{i}: {choice}" for i, choice in enumerate(choices)]
    )

    prompt = "<image>\n"
    prompt += "Answer the science multiple-choice question.\n"
    prompt += "Look carefully at the image when it is relevant.\n"
    prompt += "Use the provided lecture and hint only if helpful.\n"
    prompt += "Return only the number of the correct choice.\n\n"

    grade = clean_text(row.get("grade", ""))
    subject = clean_text(row.get("subject", ""))
    topic = clean_text(row.get("topic", ""))
    category = clean_text(row.get("category", ""))
    skill = clean_text(row.get("skill", ""))

    if grade:
        prompt += f"Grade: {grade}\n"
    if subject:
        prompt += f"Subject: {subject}\n"
    if topic:
        prompt += f"Topic: {topic}\n"
    if category:
        prompt += f"Category: {category}\n"
    if skill:
        prompt += f"Skill: {skill}\n"

    lecture = clean_text(row.get("lecture", ""))

    if PROMPT_STYLE == "no_lecture":
        lecture = ""
    elif PROMPT_STYLE == "lecture_100":
        lecture = truncate_words(lecture, max_words=100)
    elif PROMPT_STYLE == "lecture_180":
        lecture = truncate_words(lecture, max_words=180)
    elif PROMPT_STYLE == "lecture_250":
        lecture = truncate_words(lecture, max_words=250)
    else:
        raise ValueError(f"Unknown PROMPT_STYLE: {PROMPT_STYLE}")

    if lecture:
        prompt += f"\nLecture:\n{lecture}\n"

    hint = clean_text(row.get("hint", ""))
    if hint:
        prompt += f"\nHint:\n{hint}\n"

    prompt += f"\nQuestion:\n{clean_text(row['question'])}\n\n"
    prompt += f"Choices:\n{choice_lines}\n\n"
    prompt += "Answer:"

    return prompt

---
### Model Loading

In [17]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
)

/home/devvingduo/miniforge3/envs/pixels-to-predictions/lib/python3.10/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


In [18]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj",
        "up_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, peft_config)
model.to(device)
model.print_trainable_parameters()

trainable params: 3,883,008 || all params: 511,365,312 || trainable%: 0.7593


---
### Scoring

In [25]:
@torch.no_grad()
def score_indices_for_row(row, prompt_builder, score_mode="mean"):
    image = Image.open(IMAGE_ROOT / row["image_path"]).convert("RGB")
    prompt = prompt_builder(row)

    num_choices = int(row["num_choices"])
    answer_texts = [f" {i}" for i in range(num_choices)]
    full_texts = [prompt + ans for ans in answer_texts]

    prompt_inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
        padding=True,
    )

    prompt_len = prompt_inputs["input_ids"].shape[1]

    full_inputs = processor(
        text=full_texts,
        images=[image] * num_choices,
        return_tensors="pt",
        padding=True,
    )

    full_inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in full_inputs.items()
    }

    input_ids = full_inputs["input_ids"]
    attention_mask = full_inputs.get("attention_mask", torch.ones_like(input_ids))

    outputs = model(**full_inputs)
    logits = outputs.logits.float()

    shift_logits = logits[:, :-1, :]
    shift_input_ids = input_ids[:, 1:]
    shift_attention = attention_mask[:, 1:]

    log_probs = F.log_softmax(shift_logits, dim=-1)

    token_log_probs = log_probs.gather(
        dim=-1,
        index=shift_input_ids.unsqueeze(-1),
    ).squeeze(-1)

    answer_mask = torch.zeros_like(shift_input_ids, dtype=torch.bool)

    start = max(prompt_len - 1, 0)
    answer_mask[:, start:] = True

    answer_mask = answer_mask & shift_attention.bool()

    scores = []

    for i in range(num_choices):
        vals = token_log_probs[i][answer_mask[i]]

        if vals.numel() == 0:
            scores.append(-1e9)
        elif score_mode == "sum":
            scores.append(vals.sum().item())
        else:
            scores.append(vals.mean().item())

    return scores

In [26]:
@torch.no_grad()
def score_text(model, processor, image, prompt, choices, score_mode="mean"):
    num_choices = len(choices)

    full_texts = [prompt + " " + c for c in choices]

    inputs = processor(
        text=full_texts,
        images=[image] * num_choices,
        return_tensors="pt",
        padding=True,
    )

    inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    input_ids = inputs["input_ids"]
    attention_mask = inputs.get("attention_mask", torch.ones_like(input_ids))

    outputs = model(**inputs)
    logits = outputs.logits.float()

    shift_logits = logits[:, :-1, :]
    shift_input_ids = input_ids[:, 1:]
    shift_attention = attention_mask[:, 1:]

    log_probs = F.log_softmax(shift_logits, dim=-1)

    token_log_probs = log_probs.gather(
        dim=-1,
        index=shift_input_ids.unsqueeze(-1),
    ).squeeze(-1)

    # isolate answer portion
    prompt_inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
        padding=True,
    )
    prompt_len = prompt_inputs["input_ids"].shape[1]

    answer_mask = torch.zeros_like(shift_input_ids, dtype=torch.bool)
    start = max(prompt_len - 1, 0)
    answer_mask[:, start:] = True
    answer_mask = answer_mask & shift_attention.bool()

    scores = []

    for i in range(num_choices):
        vals = token_log_probs[i][answer_mask[i]]

        if vals.numel() == 0:
            scores.append(-1e9)
        elif score_mode == "sum":
            scores.append(vals.sum().item())
        else:
            scores.append(vals.mean().item())

    return scores

In [35]:
def predict_row(row):
    scores = score_indices_for_row(
        row,
        prompt_builder=build_prompt,
        score_mode="mean",
    )
    pred = int(np.argmax(scores))
    return pred, scores

In [36]:
def predict_row_hybrid(row, alpha=0.85):
    image = load_image(IMAGE_ROOT / row["image_path"])
    prompt = build_prompt(row)
    choices = row["choices_list"]

    idx_scores = score_indices_for_row(row, build_prompt, "mean")
    txt_scores = score_text(model, processor, image, prompt, choices)

    idx_scores = np.array(idx_scores)
    txt_scores = np.array(txt_scores)

    idx_scores = (idx_scores - idx_scores.mean()) / (idx_scores.std() + 1e-6)
    txt_scores = (txt_scores - txt_scores.mean()) / (txt_scores.std() + 1e-6)

    combined = alpha * idx_scores + (1 - alpha) * txt_scores

    pred = int(np.argmax(combined))
    return pred, combined.tolist()

---
### Evaluation

In [37]:
subset = val_df.sample(100, random_state=42)

val_preds = []

for _, row in tqdm(subset.iterrows(), total=len(subset)):
    pred, _ = predict_row(row)
    val_preds.append(pred)

val_acc = np.mean(np.array(val_preds) == subset["answer"].values)
print("DEBUG ACC:", val_acc)

  0%|          | 0/100 [00:00<?, ?it/s]

DEBUG ACC: 0.55


In [38]:
val_preds = []
val_scores = []

for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
    pred, scores = predict_row(row)

    val_preds.append(pred)

    val_scores.append({
        "id": row["id"],
        "scores": scores,
        "num_choices": int(row["num_choices"]),
    })

val_acc = np.mean(np.array(val_preds) == val_df["answer"].values)
print("Validation accuracy:", val_acc)

  0%|          | 0/1048 [00:00<?, ?it/s]

Validation accuracy: 0.5954198473282443


In [39]:
with open("../runs/model8_val_scores.pkl", "wb") as f:
    pickle.dump(val_scores, f)

print("Saved Model 8 val scores")

Saved Model 8 val scores


In [40]:
def load_scores(path):
    with open(path, "rb") as f:
        return pickle.load(f)

scores7_val = load_scores("../runs/model7_lecture_180_attn_r8_mean_val_scores.pkl")
scores8_val = load_scores("../runs/model8_val_scores.pkl")

labels = val_df["answer"].values

In [46]:
scores7_fixed = []

for i, row in val_df.iterrows():
    scores7_fixed.append({
        "id": row["id"],
        "scores": scores7_list[i],
        "num_choices": int(row["num_choices"]),
    })

scores7_list = scores7_fixed

In [47]:
scores7_dict = {x["id"]: x for x in scores7_list}
scores8_dict = {x["id"]: x for x in scores8_list}

Run weight search again.

In [49]:
best_acc = 0
best_w = 0

for w in np.linspace(0.7, 1.0, 21):  # bias toward model 7
    preds = []

    for item7 in scores7_list:
        id_ = item7["id"]
        item8 = scores8_dict[id_]

        s7 = np.array(item7["scores"])
        s8 = np.array(item8["scores"])

        n = item7["num_choices"]

        s7 = (s7[:n] - s7[:n].mean()) / (s7[:n].std() + 1e-6)
        s8 = (s8[:n] - s8[:n].mean()) / (s8[:n].std() + 1e-6)

        combined = w * s7 + (1 - w) * s8
        preds.append(np.argmax(combined))

    acc = np.mean(np.array(preds) == labels)

    print(f"w7={w:.2f} → acc={acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        best_w = w

print("\nBEST:")
print("Model 7 weight:", best_w)
print("Model 8 weight:", 1 - best_w)
print("Validation accuracy:", best_acc)

w7=0.70 → acc=0.6689
w7=0.71 → acc=0.6746
w7=0.73 → acc=0.6737
w7=0.74 → acc=0.6775
w7=0.76 → acc=0.6765
w7=0.77 → acc=0.6765
w7=0.79 → acc=0.6794
w7=0.80 → acc=0.6803
w7=0.82 → acc=0.6803
w7=0.83 → acc=0.6794
w7=0.85 → acc=0.6794
w7=0.86 → acc=0.6794
w7=0.88 → acc=0.6794
w7=0.90 → acc=0.6794
w7=0.91 → acc=0.6813
w7=0.93 → acc=0.6842
w7=0.94 → acc=0.6832
w7=0.96 → acc=0.6832
w7=0.97 → acc=0.6842
w7=0.98 → acc=0.6832
w7=1.00 → acc=0.6842

BEST:
Model 7 weight: 0.925
Model 8 weight: 0.07499999999999996
Validation accuracy: 0.6841603053435115


No improvement after assembling.